# Shor Period Finding with Strict Single-Shot Contextual Error Correction

This notebook demonstrates Shor period finding for $N=15$ and base $a=2$ while explicitly separating three different sources of change:

$$
\text{legitimate algorithmic evolution}
\neq
\text{contextual injected error}
\neq
\text{single-shot sampling deviation}.
$$

The implementation uses an independently propagated clean trajectory as the contextual expected reference. The noisy trajectory receives coherent contextual errors **inside circuit depth**, after legitimate operations. Therefore, an early injected error becomes the input to later controlled modular multiplications and inverse-QFT gates and is genuinely propagated by the remaining circuit.

The strict measurement layer uses

$$
\mathrm{SHOTS}=1.
$$

Exact statevectors and exact probability distributions remain available only as simulation-side **expected-reference and audit objects**. They are not presented as single-shot observations.

The correction stage is explicitly isolated from the injection parameters. It does not receive the error map, injected rotation angles, noise seed, or exact noisy expectation. Its measurement-level inputs are only the observed one-shot vector and the independently propagated ideal contextual expectation.

In [ ]:
!pip install --quiet numpy pennylane pennylane-lightning[gpu]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.5/924.5 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 869.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.5/366.

In [ ]:
import numpy as np
import pennylane as qml
from fractions import Fraction
from math import gcd

SEED = 42
NOISE_SEED = SEED + 15015
rng_noise = np.random.default_rng(NOISE_SEED)

N = 15
a = 2

# Four counting qubits are enough for this controlled N=15, a=2 demonstration.
N_COUNT = 4
N_WORK = 4
N_QUBITS = N_COUNT + N_WORK

COUNT_WIRES = list(range(N_COUNT))
WORK_WIRES = list(range(N_COUNT, N_QUBITS))

SHOTS = 1
EPS = 1e-12

RX_ERROR_RANGE = (-0.030, 0.030)
RY_ERROR_RANGE = (-0.040, 0.040)
RZ_ERROR_RANGE = (-0.030, 0.030)

try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    BACKEND = "lightning.qubit"
except Exception:
    dev = qml.device("default.qubit", wires=N_QUBITS)
    BACKEND = "default.qubit"

print("=" * 82)
print("SHOR CONFIGURATION")
print("=" * 82)
print(f"N                    : {N}")
print(f"Base a               : {a}")
print(f"Counting qubits      : {N_COUNT}")
print(f"Work qubits          : {N_WORK}")
print(f"Total qubits         : {N_QUBITS}")
print(f"Expected period      : 4")
print(f"Backend              : {BACKEND}")
print(f"Strict measurement shots : {SHOTS}")

SHOR CONFIGURATION
N                    : 15
Base a               : 2
Counting qubits      : 4
Work qubits          : 4
Total qubits         : 8
Expected period      : 4
Backend              : lightning.qubit
Strict measurement shots : 1


## Controlled modular multiplication

For Shor period finding, the work register is initialized to $|1\rangle$. The counting register controls modular multiplication by powers

$$
a^{2^k}\pmod N.
$$

For this demonstration,

$$
N=15,\qquad a=2,
$$

and the relevant multipliers are generated algorithmically rather than supplied to the period-inference stage.

The controlled unitary acts as

$$
|c\rangle|y\rangle
\longmapsto
|c\rangle
\left|
a^{c\,2^k}y\bmod N
\right\rangle.
$$

These modular operations are **legitimate Shor evolution**. Their effect on the quantum state must therefore be included in the ideal contextual trajectory and must never be classified as hardware error.

In [ ]:

def modular_multiplication_matrix(multiplier, N=15, dim=16):
    U = np.zeros((dim, dim), dtype=np.complex128)
    for y in range(dim):
        if y < N:
            out = (multiplier * y) % N
        else:
            out = y
        U[out, y] = 1.0

    if not np.allclose(U.conj().T @ U, np.eye(dim), atol=1e-12):
        raise ValueError("Modular multiplication matrix is not unitary.")
    return U

MODULAR_UNITS = []
for k in range(N_COUNT):
    multiplier = pow(a, 2 ** k, N)
    MODULAR_UNITS.append(modular_multiplication_matrix(multiplier, N=N))
    print(f"k={k} | 2^(2^{k}) mod {N} = {multiplier}")


k=0 | 2^(2^0) mod 15 = 2
k=1 | 2^(2^1) mod 15 = 4
k=2 | 2^(2^2) mod 15 = 1
k=3 | 2^(2^3) mod 15 = 1


## Explicit inverse quantum Fourier transform

After phase accumulation, the counting register is transformed by the inverse QFT.

For $Q=2^{n_c}$ counting states,

$$
U_{\mathrm{QFT}}^{-1}|x\rangle
=
\frac{1}{\sqrt{Q}}
\sum_{y=0}^{Q-1}
e^{-2\pi ixy/Q}|y\rangle.
$$

The inverse QFT is decomposed explicitly into SWAP, controlled-phase, and Hadamard operations. This is important for the contextual-error experiment because errors can be injected **between** legitimate inverse-QFT gates rather than appended only after the complete algorithm.

A legitimate inverse-QFT displacement is therefore part of the expected algorithmic trajectory:

$$
D_l^{\mathrm{legit}}
=
|\psi_l^{\mathrm{ideal}}\rangle
-
|\psi_{l-1}^{\mathrm{ideal}}\rangle.
$$

It is not an error term.

In [ ]:

def inverse_qft_ops(wires):
    ops = []

    # Reverse register order first.
    for i in range(len(wires) // 2):
        ops.append(("SWAP", (wires[i], wires[-i - 1]), 0.0))

    # Inverse of the standard QFT gate sequence.
    for j in reversed(range(len(wires))):
        for k in reversed(range(j + 1, len(wires))):
            angle = -np.pi / (2 ** (k - j))
            ops.append(("CPHASE", (wires[k], wires[j]), angle))
        ops.append(("H", (wires[j],), 0.0))

    return ops

IQFT_OPS = inverse_qft_ops(COUNT_WIRES)

def apply_iqft_op(op):
    kind, wires, angle = op
    if kind == "H":
        qml.Hadamard(wires=wires[0])
    elif kind == "CPHASE":
        qml.ControlledPhaseShift(angle, wires=list(wires))
    elif kind == "SWAP":
        qml.SWAP(wires=list(wires))
    else:
        raise ValueError(kind)

print(f"Explicit inverse-QFT operations: {len(IQFT_OPS)}")


Explicit inverse-QFT operations: 12


## Contextual coherent error injection

The noisy branch receives a deterministic simulation-side contextual error operator after each audited circuit operation.

At context $l$,

$$
E_l
=
\prod_q
R_Z(\epsilon^Z_{lq})
R_Y(\epsilon^Y_{lq})
R_X(\epsilon^X_{lq}).
$$

The clean trajectory evolves as

$$
|\psi_l^{\mathrm{ideal}}\rangle
=
U_l|\psi_{l-1}^{\mathrm{ideal}}\rangle,
$$

whereas the noisy trajectory evolves as

$$
|\psi_l^{\mathrm{noisy}}\rangle
=
E_l U_l|\psi_{l-1}^{\mathrm{noisy}}\rangle.
$$

Consequently,

$$
|\psi_L^{\mathrm{noisy}}\rangle
=
E_LU_L\cdots E_2U_2E_1U_1|\psi_0\rangle.
$$

An error introduced at an early depth is therefore transformed by every later Shor operation. The error is not modeled as a final output offset.

The error map is part of the **injection branch only**. Later correction code is not allowed to read this map.

In [ ]:

# One error triplet per contextual injection point and qubit.
# Contexts:
#   0..N_COUNT-1                       : Hadamards
#   N_COUNT..2*N_COUNT-1               : controlled modular multiplications
#   remaining                          : inverse-QFT operations

N_CONTEXTS = 2 * N_COUNT + len(IQFT_OPS)

ERROR_MAP = np.zeros((N_CONTEXTS, N_QUBITS, 3), dtype=np.float64)
ERROR_MAP[:, :, 0] = rng_noise.uniform(*RX_ERROR_RANGE, size=(N_CONTEXTS, N_QUBITS))
ERROR_MAP[:, :, 1] = rng_noise.uniform(*RY_ERROR_RANGE, size=(N_CONTEXTS, N_QUBITS))
ERROR_MAP[:, :, 2] = rng_noise.uniform(*RZ_ERROR_RANGE, size=(N_CONTEXTS, N_QUBITS))

def inject_contextual_error(context_index):
    for q in range(N_QUBITS):
        ex, ey, ez = ERROR_MAP[context_index, q]
        qml.RX(ex, wires=q)
        qml.RY(ey, wires=q)
        qml.RZ(ez, wires=q)

print("=" * 82)
print("CONTEXTUAL ERROR MAP")
print("=" * 82)
print(f"Contexts             : {N_CONTEXTS}")
print(f"Noise seed           : {NOISE_SEED}")
print(f"RX observed range    : [{ERROR_MAP[:,:,0].min():+.5f}, {ERROR_MAP[:,:,0].max():+.5f}]")
print(f"RY observed range    : [{ERROR_MAP[:,:,1].min():+.5f}, {ERROR_MAP[:,:,1].max():+.5f}]")
print(f"RZ observed range    : [{ERROR_MAP[:,:,2].min():+.5f}, {ERROR_MAP[:,:,2].max():+.5f}]")


CONTEXTUAL ERROR MAP
Contexts             : 20
Noise seed           : 15057
RX observed range    : [-0.02989, +0.02979]
RY observed range    : [-0.03976, +0.03924]
RZ observed range    : [-0.02994, +0.02981]


## Independent ideal and noisy Shor trajectories

Both trajectories begin from the same legitimate Shor input:

$$
|0\rangle^{\otimes n_c}\otimes|1\rangle.
$$

They then receive the same Hadamards, the same controlled modular multiplications, and the same inverse-QFT operations.

The ideal trajectory contains no injected contextual perturbation. The noisy trajectory receives the contextual operator after each audited operation.

At identical circuit depth, the contextual state error is defined as

$$
D_l^{\mathrm{error}}
=
|\psi_l^{\mathrm{noisy}}\rangle
-
|\psi_l^{\mathrm{ideal}}\rangle.
$$

This **same-depth comparison** is essential: state changes produced by input preparation, modular exponentiation, phase accumulation, or inverse-QFT evolution remain inside the ideal reference and are therefore not mislabeled as error.

In [ ]:
def prepare_work_one():
    # WORK_WIRES[-1] is the least-significant work bit under PennyLane basis ordering here.
    qml.PauliX(wires=WORK_WIRES[-1])

def apply_controlled_modular_unitary(control_wire, U):
    qml.ControlledQubitUnitary(
        U,
        wires=[control_wire] + WORK_WIRES
    )

@qml.qnode(dev)
def shor_ideal_pre_iqft():
    prepare_work_one()

    for wire in COUNT_WIRES:
        qml.Hadamard(wires=wire)

    # Counting bit significance is reversed relative to k here.
    for k, U in enumerate(MODULAR_UNITS):
        control_wire = COUNT_WIRES[-1 - k]
        apply_controlled_modular_unitary(control_wire, U)

    return qml.state()

@qml.qnode(dev)
def shor_ideal_final():
    prepare_work_one()

    for wire in COUNT_WIRES:
        qml.Hadamard(wires=wire)

    for k, U in enumerate(MODULAR_UNITS):
        control_wire = COUNT_WIRES[-1 - k]
        apply_controlled_modular_unitary(control_wire, U)

    for op in IQFT_OPS:
        apply_iqft_op(op)

    return qml.state()

@qml.qnode(dev)
def shor_noisy_final():
    prepare_work_one()

    context = 0

    for wire in COUNT_WIRES:
        qml.Hadamard(wires=wire)
        inject_contextual_error(context)
        context += 1

    for k, U in enumerate(MODULAR_UNITS):
        control_wire = COUNT_WIRES[-1 - k]
        apply_controlled_modular_unitary(control_wire, U)
        inject_contextual_error(context)
        context += 1

    for op in IQFT_OPS:
        apply_iqft_op(op)
        inject_contextual_error(context)
        context += 1

    return qml.state()


# ---------------------------------------------------------------------
# Independent stage-by-stage propagation audit
# ---------------------------------------------------------------------
def apply_single_operation_to_state(state, operation_callable):
    audit_dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(audit_dev)
    def circuit():
        qml.StatePrep(state, wires=range(N_QUBITS))
        operation_callable()
        return qml.state()
    return np.asarray(circuit(), dtype=np.complex128)

def apply_context_error_to_state(state, context_index):
    audit_dev = qml.device("default.qubit", wires=N_QUBITS)
    @qml.qnode(audit_dev)
    def circuit():
        qml.StatePrep(state, wires=range(N_QUBITS))
        inject_contextual_error(context_index)
        return qml.state()
    return np.asarray(circuit(), dtype=np.complex128)

# Build the legitimate input |0...0>|1>.
dim = 2 ** N_QUBITS
initial_state = np.zeros(dim, dtype=np.complex128)
# WORK_WIRES[-1] is the least-significant wire; basis index 1 represents work |1>.
initial_state[1] = 1.0

ideal_stage = initial_state.copy()
noisy_stage = initial_state.copy()
stage_records = []
context_index = 0

def audit_stage(label, operation_callable, context_index):
    global ideal_stage, noisy_stage

    previous_ideal = ideal_stage.copy()
    ideal_stage = apply_single_operation_to_state(ideal_stage, operation_callable)

    noisy_stage = apply_single_operation_to_state(noisy_stage, operation_callable)
    noisy_stage = apply_context_error_to_state(noisy_stage, context_index)

    # Global phase alignment is audit-only and changes no measurement probability.
    overlap = np.vdot(ideal_stage, noisy_stage)
    if abs(overlap) > EPS:
        noisy_stage = noisy_stage * np.conj(overlap / abs(overlap))

    legitimate_rms = float(np.sqrt(np.mean(np.abs(ideal_stage - previous_ideal) ** 2)))
    error_rms = float(np.sqrt(np.mean(np.abs(noisy_stage - ideal_stage) ** 2)))
    fidelity = float(np.abs(np.vdot(ideal_stage, noisy_stage)) ** 2)

    stage_records.append((label, legitimate_rms, error_rms, fidelity))

for wire in COUNT_WIRES:
    audit_stage(
        f"H q{wire}",
        lambda wire=wire: qml.Hadamard(wires=wire),
        context_index
    )
    context_index += 1

for k, U in enumerate(MODULAR_UNITS):
    control_wire = COUNT_WIRES[-1-k]
    audit_stage(
        f"CMULT k={k}",
        lambda control_wire=control_wire, U=U:
            apply_controlled_modular_unitary(control_wire, U),
        context_index
    )
    context_index += 1

for iqft_index, op in enumerate(IQFT_OPS):
    audit_stage(
        f"IQFT {iqft_index+1}:{op[0]}",
        lambda op=op: apply_iqft_op(op),
        context_index
    )
    context_index += 1

print("=" * 112)
print("STAGE-BY-STAGE EXPECTED-TRAJECTORY / ERROR-PROPAGATION AUDIT")
print("=" * 112)
print(f"{'Stage':<22}{'Legitimate RMS':>20}{'Same-depth error RMS':>26}{'Ideal/noisy fidelity':>24}")
print("-" * 112)
for label, legit_rms_stage, err_rms_stage, fid_stage in stage_records:
    print(f"{label:<22}{legit_rms_stage:>20.8e}{err_rms_stage:>26.8e}{fid_stage:>24.10f}")
print("-" * 112)
print("Legitimate RMS is algorithmic evolution. Same-depth error RMS is noisy-vs-ideal.")
print("Earlier injected errors remain inside the noisy state and propagate through later operations.")

STAGE-BY-STAGE EXPECTED-TRAJECTORY / ERROR-PROPAGATION AUDIT
Stage                       Legitimate RMS      Same-depth error RMS    Ideal/noisy fidelity
----------------------------------------------------------------------------------------------------------------
H q0                        4.78354290e-02            2.69601115e-03            0.9981401357
H q1                        4.78354290e-02            3.73344349e-03            0.9964349015
H q2                        4.78354290e-02            3.64067384e-03            0.9966097248
H q3                        4.78354290e-02            4.83695525e-03            0.9940195574
CMULT k=0                   6.25000000e-02            5.48062286e-03            0.9923252521
CMULT k=1                   6.25000000e-02            6.64483775e-03            0.9887285513
CMULT k=2                   0.00000000e+00            7.37591463e-03            0.9861210397
CMULT k=3                   0.00000000e+00            6.95296454e-03            0.

## Contextual state correction and leakage boundary

The exact statevector branch is retained as an **expected-state audit**. After global-phase alignment, the final contextual state displacement is

$$
\Delta_{\mathrm{context}}
=
|\psi^{\mathrm{noisy}}\rangle
-
|\psi^{\mathrm{ideal}}\rangle.
$$

For the expected-state consistency audit, deterministic reference projection gives

$$
|\psi^{\mathrm{corr}}\rangle
=
|\psi^{\mathrm{noisy}}\rangle
-
\Delta_{\mathrm{context}}
=
|\psi^{\mathrm{ideal}}\rangle.
$$

This exact-state operation is not claimed to be a one-shot hardware measurement procedure. The strict one-shot correction is implemented later at the measurement-distribution level.

The important scientific distinction is that the ideal reference already contains all legitimate Shor evolution. Thus the correction target does not attempt to remove modular exponentiation, phase accumulation, or inverse-QFT evolution.

The injection and correction branches are kept logically separate. The strict correction function later receives only

$$
M^{(1)}
\quad\text{and}\quad
E^{\mathrm{ideal}},
$$

and receives no `ERROR_MAP`, injected angles, `NOISE_SEED`, injection function, or exact noisy expectation.

In [ ]:

def normalize_state(state):
    state = np.asarray(state, dtype=np.complex128)
    n = np.linalg.norm(state)
    if n < EPS:
        raise ValueError("Zero state.")
    return state / n

def align_global_phase(reference, state):
    overlap = np.vdot(reference, state)
    if abs(overlap) < EPS:
        return state
    phase = overlap / abs(overlap)
    return state * np.conj(phase)

pre_iqft = normalize_state(shor_ideal_pre_iqft())
ideal = normalize_state(shor_ideal_final())
noisy = normalize_state(shor_noisy_final())

noisy_aligned = align_global_phase(ideal, noisy)

legitimate_iqft_displacement = ideal - pre_iqft
context_error = noisy_aligned - ideal

# Same deterministic contextual-reference projection used in the earlier notebooks.
corrected = noisy_aligned - context_error
corrected = normalize_state(corrected)

post_correction_residual = corrected - ideal

iqft_rms = np.sqrt(np.mean(np.abs(legitimate_iqft_displacement) ** 2))
error_rms = np.sqrt(np.mean(np.abs(context_error) ** 2))
post_rms = np.sqrt(np.mean(np.abs(post_correction_residual) ** 2))

num = np.real(np.vdot(legitimate_iqft_displacement, context_error))
den = np.linalg.norm(legitimate_iqft_displacement) * np.linalg.norm(context_error) + EPS
alignment = float(num / den)

print("=" * 82)
print("SHOR / CONTEXTUAL ERROR DECOMPOSITION")
print("=" * 82)
print(f"Legitimate inverse-QFT displacement RMS : {iqft_rms:.8e}")
print(f"Context-propagated error RMS            : {error_rms:.8e}")
print(f"Post-correction residual RMS            : {post_rms:.8e}")
print(f"IQFT↔error real alignment               : {alignment:+.8f}")
print("Inverse-QFT evolution is retained; noisy-vs-ideal displacement is corrected.")


SHOR / CONTEXTUAL ERROR DECOMPOSITION
Legitimate inverse-QFT displacement RMS : 7.65465545e-02
Context-propagated error RMS            : 8.89535128e-03
Post-correction residual RMS            : 1.38798005e-17
IQFT↔error real alignment               : +0.01224718
Inverse-QFT evolution is retained; noisy-vs-ideal displacement is corrected.


## Counting-register expected values

The work register is traced out by summing its basis probabilities, producing the expected counting-register distribution

$$
E^{\mathrm{ideal}}(y)
=
P_{\mathrm{ideal}}(y),
$$

and the audit-only noisy expectation

$$
E^{\mathrm{noisy}}(y)
=
P_{\mathrm{noisy}}(y).
$$

The noisy-vs-ideal expected displacement is

$$
\Delta_{\mathrm{context}}(y)
=
E^{\mathrm{noisy}}(y)
-
E^{\mathrm{ideal}}(y).
$$

These exact distributions are simulation-side expectations. The actual strict measurement diagnostic in the next section samples exactly one outcome.

In [ ]:
def counting_probabilities(state):
    state = normalize_state(state)
    probs_full = np.abs(state) ** 2

    # First N_COUNT wires are the counting register, so reshape as
    # [counting basis, work basis] and sum out work.
    probs = probs_full.reshape(2 ** N_COUNT, 2 ** N_WORK).sum(axis=1)
    probs /= probs.sum()
    return probs

P_ideal = counting_probabilities(ideal)
P_noisy = counting_probabilities(noisy_aligned)
P_corrected = counting_probabilities(corrected)

mae_noisy = np.mean(np.abs(P_noisy - P_ideal))
mae_corrected = np.mean(np.abs(P_corrected - P_ideal))

recovery = 100.0 * (mae_noisy - mae_corrected) / (mae_noisy + EPS)

print("=" * 82)
print("PERIOD-DISTRIBUTION AUDIT")
print("=" * 82)
print(f"Noisy → ideal probability MAE     : {mae_noisy:.8e}")
print(f"Corrected → ideal probability MAE : {mae_corrected:.8e}")
print(f"Expected-state reference recovery     : {recovery:+.6f}%")
print()

print(" y |   phase   |      ideal |      noisy |  corrected")
print("-" * 62)
for y in range(2 ** N_COUNT):
    if max(P_ideal[y], P_noisy[y], P_corrected[y]) > 1e-4:
        print(
            f"{y:2d} | {y/(2**N_COUNT):8.5f} | "
            f"{P_ideal[y]:10.6f} | {P_noisy[y]:10.6f} | {P_corrected[y]:10.6f}"
        )

PERIOD-DISTRIBUTION AUDIT
Noisy → ideal probability MAE     : 6.19061303e-04
Corrected → ideal probability MAE : 7.17197151e-36
Expected-state reference recovery     : +100.000000%

 y |   phase   |      ideal |      noisy |  corrected
--------------------------------------------------------------
 0 |  0.00000 |   0.250000 |   0.248191 |   0.250000
 1 |  0.06250 |   0.000000 |   0.000339 |   0.000000
 2 |  0.12500 |   0.000000 |   0.000542 |   0.000000
 3 |  0.18750 |   0.000000 |   0.000329 |   0.000000
 4 |  0.25000 |   0.250000 |   0.250059 |   0.250000
 5 |  0.31250 |   0.000000 |   0.000369 |   0.000000
 6 |  0.37500 |   0.000000 |   0.000552 |   0.000000
 7 |  0.43750 |   0.000000 |   0.000334 |   0.000000
 8 |  0.50000 |   0.250000 |   0.247552 |   0.250000
 9 |  0.56250 |   0.000000 |   0.000323 |   0.000000
10 |  0.62500 |   0.000000 |   0.000540 |   0.000000
11 |  0.68750 |   0.000000 |   0.000327 |   0.000000
12 |  0.75000 |   0.250000 |   0.249304 |   0.250000
13 |  0.8125

## Strict single-shot measurement, residual decomposition, and correction

The counting register is now measured with exactly one shot:

$$
M^{(1)}(y)\in\{0,1\},
\qquad
\sum_y M^{(1)}(y)=1.
$$

For audit purposes only, the total one-shot residual can be decomposed as

$$
R(y)
=
M^{(1)}(y)-E^{\mathrm{ideal}}(y)
$$

and

$$
R(y)
=
\underbrace{
E^{\mathrm{noisy}}(y)-E^{\mathrm{ideal}}(y)
}_{\Delta_{\mathrm{context}}(y)}
+
\underbrace{
M^{(1)}(y)-E^{\mathrm{noisy}}(y)
}_{S^{(1)}(y)}.
$$

The first term is the expected contextual circuit displacement. The second term is the stochastic single-shot sampling displacement.

### Leakage boundary

The correction function itself receives only

$$
M^{(1)}
\quad\text{and}\quad
E^{\mathrm{ideal}}.
$$

It does **not** receive $E^{\mathrm{noisy}}$, `ERROR_MAP`, the injected $R_X/R_Y/R_Z$ angles, `NOISE_SEED`, or `inject_contextual_error()`.

The deterministic reference correction is

$$
R_{\mathrm{corr}}
=
M^{(1)}-E^{\mathrm{ideal}},
$$

$$
M^{\mathrm{corr}}
=
M^{(1)}-R_{\mathrm{corr}}
=
E^{\mathrm{ideal}}.
$$

The reported correction percentage is

$$
C=
100
\left(
1-
\frac{
\operatorname{MAE}(M^{\mathrm{corr}},E^{\mathrm{ideal}})
}{
\operatorname{MAE}(M^{(1)},E^{\mathrm{ideal}})
}
\right).
$$

Because this is a deterministic projection onto a supplied contextual ideal reference, 100% reference recovery is expected up to numerical precision. It must not be interpreted as blind inference of an unknown physical hardware error from one shot.

In [ ]:
shot_rng = np.random.default_rng(SEED + 909)

def one_shot_vector(probs):
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum()
    outcome = int(shot_rng.choice(len(probs), size=1, p=probs)[0])
    measured = np.zeros_like(probs)
    measured[outcome] = 1.0
    return outcome, measured

def contextual_reference_correction(measured_one_shot, ideal_expected):
    # STRICT LEAKAGE BOUNDARY:
    # No ERROR_MAP, injected angle, NOISE_SEED, injection function,
    # or noisy expected distribution enters this function.
    residual = measured_one_shot - ideal_expected
    corrected_measurement = measured_one_shot - residual
    return corrected_measurement, residual

ideal_outcome, M_ideal_1 = one_shot_vector(P_ideal)
noisy_outcome, M_noisy_1 = one_shot_vector(P_noisy)

# AUDIT-ONLY decomposition. P_noisy is never passed to correction.
hardware_context_expected = P_noisy - P_ideal
single_shot_sampling_residual = M_noisy_1 - P_noisy
total_one_shot_residual = M_noisy_1 - P_ideal
decomposition_closure = total_one_shot_residual - (
    hardware_context_expected + single_shot_sampling_residual
)

# Strict correction boundary.
M_corrected, correction_residual = contextual_reference_correction(
    M_noisy_1,
    P_ideal
)

pre_correction_mae = float(np.mean(np.abs(M_noisy_1 - P_ideal)))
post_correction_mae = float(np.mean(np.abs(M_corrected - P_ideal)))

if pre_correction_mae > EPS:
    correction_percentage = 100.0 * (
        1.0 - post_correction_mae / pre_correction_mae
    )
else:
    correction_percentage = 100.0 if post_correction_mae <= EPS else 0.0

print("=" * 94)
print("STRICT SINGLE-SHOT RESIDUAL DECOMPOSITION")
print("=" * 94)
print(f"Shots per measured trajectory        : {SHOTS}")
print(f"Ideal one-shot outcome               : y={ideal_outcome:2d} | phase={ideal_outcome/(2**N_COUNT):.5f}")
print(f"Noisy one-shot outcome               : y={noisy_outcome:2d} | phase={noisy_outcome/(2**N_COUNT):.5f}")
print(f"Context/hardware expected MAE        : {np.mean(np.abs(hardware_context_expected)):.8e}")
print(f"Single-shot sampling residual MAE    : {np.mean(np.abs(single_shot_sampling_residual)):.8e}")
print(f"Total one-shot residual MAE          : {np.mean(np.abs(total_one_shot_residual)):.8e}")
print(f"Residual decomposition closure max Δ : {np.max(np.abs(decomposition_closure)):.8e}")
print(f"Pre-correction → ideal MAE           : {pre_correction_mae:.8e}")
print(f"Post-correction → ideal MAE          : {post_correction_mae:.8e}")
print(f"ERROR CORRECTION PERCENTAGE          : {correction_percentage:.6f}%")
print()
print("CORRECTION INPUT BOUNDARY")
print("-" * 94)
print("Correction inputs                       : M_noisy_1, P_ideal")
print("ERROR_MAP passed to correction           : NO")
print("Injected RX/RY/RZ angles passed          : NO")
print("NOISE_SEED passed to correction          : NO")
print("P_noisy passed to correction             : NO (audit-only)")

STRICT SINGLE-SHOT RESIDUAL DECOMPOSITION
Shots per measured trajectory        : 1
Ideal one-shot outcome               : y=12 | phase=0.75000
Noisy one-shot outcome               : y= 4 | phase=0.25000
Context/hardware expected MAE        : 6.19061303e-04
Single-shot sampling residual MAE    : 9.37425658e-02
Total one-shot residual MAE          : 9.37500000e-02
Residual decomposition closure max Δ : 1.39426505e-34
Pre-correction → ideal MAE           : 9.37500000e-02
Post-correction → ideal MAE          : 0.00000000e+00
ERROR CORRECTION PERCENTAGE          : 100.000000%

CORRECTION INPUT BOUNDARY
----------------------------------------------------------------------------------------------
Correction inputs                       : M_noisy_1, P_ideal
ERROR_MAP passed to correction           : NO
Injected RX/RY/RZ angles passed          : NO
NOISE_SEED passed to correction          : NO
P_noisy passed to correction             : NO (audit-only)


## Blind period inference and classical factor extraction

For a measured counting value $y$, the phase estimate is

$$
\phi=\frac{y}{2^{n_c}}.
$$

A continued-fraction approximation is used to obtain a candidate denominator. Candidate multiples are then tested using the modular condition

$$
a^r\equiv1\pmod N.
$$

Once an even period $r$ is obtained, classical factor extraction uses

$$
p=\gcd\left(a^{r/2}-1,N\right),
$$

$$
q=\gcd\left(a^{r/2}+1,N\right).
$$

The inference function is not supplied the known period $r=4$.

Under a strict single shot, the raw noisy measurement may or may not contain enough information to infer the period. This is an inherent statistical limitation of one-shot period finding, not a correction-code failure.

The corrected measurement-level reference is also evaluated. Because deterministic contextual-reference projection returns the ideal expected distribution, its inference behavior matches that reference by construction.

In [ ]:
def infer_period_from_distribution(probs, a, N, n_count):
    order = np.argsort(probs)[::-1]

    attempts = []

    for y in order:
        if y == 0 or probs[y] <= 0:
            continue

        phase = y / (2 ** n_count)
        frac = Fraction(phase).limit_denominator(N)
        denom = frac.denominator

        # A measured phase may reduce s/r to a denominator dividing r.
        # Test small multiples without supplying the true r.
        for mult in range(1, N + 1):
            r_candidate = denom * mult
            if r_candidate >= N:
                break

            valid = pow(a, r_candidate, N) == 1
            attempts.append((int(y), float(probs[y]), phase, denom, r_candidate, valid))

            if valid:
                return r_candidate, attempts

    return None, attempts

def factors_from_period(a, N, r):
    if r is None or r % 2 != 0:
        return None

    x = pow(a, r // 2, N)

    if x in (1, N - 1):
        return None

    p = gcd(x - 1, N)
    q = gcd(x + 1, N)

    factors = sorted({p, q})
    factors = [f for f in factors if 1 < f < N and N % f == 0]

    if len(factors) >= 2:
        return tuple(factors[:2])
    return None

def report_period_result(name, probs):
    r, attempts = infer_period_from_distribution(probs, a, N, N_COUNT)
    factors = factors_from_period(a, N, r)

    print(f"{name:<10} | inferred r={str(r):>4} | factors={factors}")
    return r, factors, attempts


print("=" * 82)
print("BLIND PERIOD INFERENCE — EXPECTED DISTRIBUTION AUDIT")
print("=" * 82)
r_ideal, f_ideal, _ = report_period_result("IDEAL", P_ideal)
r_noisy, f_noisy, _ = report_period_result("NOISY", P_noisy)

print()
print("=" * 82)
print("BLIND PERIOD INFERENCE — STRICT SINGLE SHOT")
print("=" * 82)
rs_ideal, fs_ideal, _ = report_period_result("IDEAL-1", M_ideal_1)
rs_noisy, fs_noisy, _ = report_period_result("NOISY-1", M_noisy_1)
rs_corr, fs_corr, _ = report_period_result("CORR", M_corrected)

BLIND PERIOD INFERENCE — EXPECTED DISTRIBUTION AUDIT
IDEAL      | inferred r=   4 | factors=(3, 5)
NOISY      | inferred r=   4 | factors=(3, 5)

BLIND PERIOD INFERENCE — STRICT SINGLE SHOT
IDEAL-1    | inferred r=   4 | factors=(3, 5)
NOISY-1    | inferred r=   4 | factors=(3, 5)
CORR       | inferred r=   4 | factors=(3, 5)


## Final scientific audit

The final audit checks the complete framework:

$$
\text{legitimate Shor evolution}
\rightarrow
\text{independent contextual expectation},
$$

$$
\text{injected contextual error}
\rightarrow
\text{propagation through later circuit depth},
$$

$$
\text{one-shot observation}
\rightarrow
\text{separate stochastic sampling residual},
$$

and

$$
\text{correction}
\rightarrow
\text{reference projection without injection-parameter access}.
$$

A decomposition closure near machine precision verifies

$$
M^{(1)}-E^{\mathrm{ideal}}
=
\left(E^{\mathrm{noisy}}-E^{\mathrm{ideal}}\right)
+
\left(M^{(1)}-E^{\mathrm{noisy}}\right).
$$

The correction percentage quantifies recovery of the supplied contextual reference. It is not a claim of independent physical quantum-error correction on unknown hardware.

In [ ]:
print("=" * 94)
print("FINAL SHOR + STRICT SINGLE-SHOT CONTEXTUAL CORRECTION SUMMARY")
print("=" * 94)
print(f"Problem                                  : factor N={N} with a={a}")
print(f"Expected mathematical period             : 4")
print(f"Measurement shots                        : {SHOTS}")
print(f"Legitimate IQFT displacement             : {iqft_rms:.8e} RMS")
print(f"Context-propagated state error            : {error_rms:.8e} RMS")
print(f"Expected-state correction residual        : {post_rms:.8e} RMS")
print(f"Expected noisy probability → ideal MAE   : {mae_noisy:.8e}")
print(f"Context/hardware expected MAE             : {np.mean(np.abs(hardware_context_expected)):.8e}")
print(f"Single-shot sampling residual MAE         : {np.mean(np.abs(single_shot_sampling_residual)):.8e}")
print(f"Total one-shot residual MAE               : {np.mean(np.abs(total_one_shot_residual)):.8e}")
print(f"Decomposition closure max Δ               : {np.max(np.abs(decomposition_closure)):.8e}")
print(f"Corrected → ideal reference MAE           : {post_correction_mae:.8e}")
print(f"ERROR CORRECTION PERCENTAGE               : {correction_percentage:.6f}%")
print(f"Ideal expected period/factors             : r={r_ideal}, factors={f_ideal}")
print(f"Noisy expected period/factors             : r={r_noisy}, factors={f_noisy}")
print(f"Ideal strict-1-shot period/factors        : r={rs_ideal}, factors={fs_ideal}")
print(f"Noisy strict-1-shot period/factors        : r={rs_noisy}, factors={fs_noisy}")
print(f"Corrected reference period/factors        : r={rs_corr}, factors={fs_corr}")
print()
print("CORRECTION INPUT BOUNDARY")
print("-" * 94)
print("Injection parameters passed to correction : NONE")
print("Exact noisy expectation passed            : NO (audit-only)")
print("Ideal contextual reference available      : YES")
print("=" * 94)

FINAL SHOR + STRICT SINGLE-SHOT CONTEXTUAL CORRECTION SUMMARY
Problem                                  : factor N=15 with a=2
Expected mathematical period             : 4
Measurement shots                        : 1
Legitimate IQFT displacement             : 7.65465545e-02 RMS
Context-propagated state error            : 8.89535128e-03 RMS
Expected-state correction residual        : 1.38798005e-17 RMS
Expected noisy probability → ideal MAE   : 6.19061303e-04
Context/hardware expected MAE             : 6.19061303e-04
Single-shot sampling residual MAE         : 9.37425658e-02
Total one-shot residual MAE               : 9.37500000e-02
Decomposition closure max Δ               : 1.39426505e-34
Corrected → ideal reference MAE           : 0.00000000e+00
ERROR CORRECTION PERCENTAGE               : 100.000000%
Ideal expected period/factors             : r=4, factors=(3, 5)
Noisy expected period/factors             : r=4, factors=(3, 5)
Ideal strict-1-shot period/factors        : r=4, factors=(3